# Геометрия реальных ридов: длины R1/R2, вставка и перекрытие — PRJEB30386 (человек, 5′MTPX, MiSeq 2×300)

Распределения снимаются с реальных данных на каждой стадии обработки, чтобы при необходимости заменить ими параметры симулятора (длина ридов, длина вставки). Симуляция по ним пока не запускается.

| стадия | что измеряется |
|---|---|
| `raw` | длины R1/R2 как вышли с секвенатора |
| `trimmed` | после снятия адаптеров и фильтра fastp |
| `pr_trimmed` | после снятия праймеров |
| `merged` | длина склеенной пары pRESTO = длина вставки (ампликон без праймеров) |
| `post_filter` | склеенные риды, прошедшие фильтр после аннотации (вход эталона симуляции) |
| `overlap` | R1 + R2 − длина склеенной вставки для одной и той же пары (`pr_trimmed` ↔ `merged`) |

Реальная библиотека не фрагментировалась: вставка здесь — целый ампликон, а не фрагмент после разрезания. Поэтому для симулятора из этих данных напрямую переносимы длины ридов и длина ампликона; распределение длин фрагментов после фрагментации задаётся моделью отбора по длине.


## Параметры

In [1]:
DATASET = "PRJEB30386"
RAW_DATASET = "PRJEB30386"
RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LABEL = {"ERR3004229": "IgM", "ERR3004230": "IgG", "ERR3004231": "IgK", "ERR3004232": "IgL"}

import gzip, csv, os
from collections import Counter
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "")), cwd, *cwd.parents,
                 Path("/data/user/epishkin"), Path("/Users/epishkin/workspace/bcr-assembler")):
        if str(root) not in ("", ".") and (root / "results" / DATASET).is_dir() and (root / "raw" / RAW_DATASET).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET} + raw/{RAW_DATASET} not found; set BCR_VOLUME")

ROOT = find_root()
RES = ROOT / "results" / DATASET
RAW = ROOT / "raw" / RAW_DATASET
OUT = RES / "qc" / "read_geometry"
OUT.mkdir(parents=True, exist_ok=True)
N_READS = 200_000   # подвыборка: первые N записей каждого файла
print(ROOT, OUT)


/Users/epishkin/workspace/bcr-assembler /Users/epishkin/workspace/bcr-assembler/results/PRJEB30386/qc/read_geometry


In [2]:
def fastq_records(path, n):
    with gzip.open(path, "rt") as h:
        for _ in range(n):
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip()
            h.readline(); h.readline()
            yield head[1:].split()[0].split("|")[0], len(seq)

def stage_files(run):
    return {
        "raw": (RAW / f"{run}_1.fastq.gz", RAW / f"{run}_2.fastq.gz"),
        "trimmed": (RES / "trimmed/fastq" / f"{run}_1.trim.fastq.gz", RES / "trimmed/fastq" / f"{run}_2.trim.fastq.gz"),
        "pr_trimmed": (RES / "pr_trimmed/fastq" / f"{run}_1.pr.fastq.gz", RES / "pr_trimmed/fastq" / f"{run}_2.pr.fastq.gz"),
        "merged": (RES / "merged/fastq" / f"{run}_assemble-pass.fastq.gz", None),
        "post_filter": (RES / "post_annotation_filtered/fastq" / f"{run}_filtered.fastq.gz", None),
    }

def quantiles(counter):
    vals = np.repeat(np.fromiter(counter.keys(), int), np.fromiter(counter.values(), int))
    if not len(vals):
        return {}
    q = np.percentile(vals, [1, 5, 25, 50, 75, 95, 99])
    return {"n": int(len(vals)), "mean": round(float(vals.mean()), 1),
            **{f"p{p}": int(v) for p, v in zip((1, 5, 25, 50, 75, 95, 99), q)}}


## Сбор распределений

In [3]:
hist = {}      # (образец, величина) -> Counter(длина)
summary = []
for run in RUNS:
    files = stage_files(run)
    pr_len = {}
    for stage, (f1, f2) in files.items():
        if not f1.exists():
            print("missing", f1); continue
        for mate, f in (("R1", f1), ("R2", f2)):
            if f is None:
                continue
            c = Counter()
            for rid, L in fastq_records(f, N_READS):
                c[L] += 1
                if stage == "pr_trimmed":
                    pr_len.setdefault(rid, {})[mate] = L
            key = stage if stage in ("merged", "post_filter") else f"{stage}_{mate}"
            hist[(run, key)] = c
            summary.append({"run": run, "label": RUN_LABEL[run], "measure": key, **quantiles(c)})
    # перекрытие R1+R2-вставка для одних и тех же пар
    ov = Counter()
    for rid, L in fastq_records(files["merged"][0], 10 * N_READS):
        p = pr_len.get(rid)
        if p and "R1" in p and "R2" in p:
            ov[p["R1"] + p["R2"] - L] += 1
    hist[(run, "overlap")] = ov
    summary.append({"run": run, "label": RUN_LABEL[run], "measure": "overlap", **quantiles(ov)})
    print(run, "done", flush=True)

fields = ["run", "label", "measure", "n", "mean", "p1", "p5", "p25", "p50", "p75", "p95", "p99"]
with open(OUT / "length_summary.tsv", "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=fields, delimiter="\t"); w.writeheader(); w.writerows(summary)
with open(OUT / "length_histograms.tsv", "w", newline="") as h:
    w = csv.writer(h, delimiter="\t"); w.writerow(["run", "measure", "length", "count"])
    for (run, m), c in sorted(hist.items()):
        for L, n in sorted(c.items()):
            w.writerow([run, m, L, n])
print("wrote", OUT / "length_summary.tsv")


ERR3004229 done


ERR3004230 done


ERR3004231 done


ERR3004232 done


wrote /Users/epishkin/workspace/bcr-assembler/results/PRJEB30386/qc/read_geometry/length_summary.tsv


## Сводка (квантили длин)

In [4]:
import pandas as pd
df = pd.read_csv(OUT / "length_summary.tsv", sep="\t")
order = ["raw_R1", "raw_R2", "trimmed_R1", "trimmed_R2", "pr_trimmed_R1", "pr_trimmed_R2", "merged", "post_filter", "overlap"]
df["measure"] = pd.Categorical(df["measure"], order, ordered=True)
df.sort_values(["measure", "run"])[["measure", "label", "n", "mean", "p5", "p25", "p50", "p75", "p95"]]


,measure,label,n,mean,p5,p25,p50,p75,p95
0,raw_R1,IgM,200000,325.0,325,325,325,325,325
9,raw_R1,IgG,200000,325.0,325,325,325,325,325
18,raw_R1,IgK,200000,325.0,325,325,325,325,325
27,raw_R1,IgL,200000,325.0,325,325,325,325,325
1,raw_R2,IgM,200000,285.0,285,285,285,285,285
10,raw_R2,IgG,200000,285.0,285,285,285,285,285
19,raw_R2,IgK,200000,285.0,285,285,285,285,285
28,raw_R2,IgL,200000,285.0,285,285,285,285,285
2,trimmed_R1,IgM,200000,325.0,325,325,325,325,325
11,trimmed_R1,IgG,200000,325.0,325,325,325,325,325


## Распределения (все run вместе)

In [5]:
panels = [("R1", ["raw_R1", "trimmed_R1", "pr_trimmed_R1"]), ("R2", ["raw_R2", "trimmed_R2", "pr_trimmed_R2"]),
          ("вставка (склеенная пара)", ["merged", "post_filter"]), ("перекрытие R1+R2−вставка", ["overlap"])]
fig, axes = plt.subplots(len(panels), 1, figsize=(10, 3.2 * len(panels)))
for ax, (title, measures) in zip(axes, panels):
    for m in measures:
        tot = Counter()
        for run in RUNS:
            tot.update(hist.get((run, m), Counter()))
        if not tot:
            continue
        xs = np.array(sorted(tot)); ys = np.array([tot[x] for x in xs], float); ys /= ys.sum()
        ax.plot(xs, ys, label=m, lw=1.4)
    ax.set_title(title); ax.set_xlabel("длина, нт"); ax.set_ylabel("доля"); ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "length_distributions.png", dpi=130)


## Как это можно использовать в симуляторе

- **Длина ридов.** Сейчас симуляция даёт ровно PE150 (профиль NovaSeq, 151 → 150). Распределения `trimmed_R1/R2` показывают, какой разброс длин оставляет реальная обрезка; при желании можно обрезать симулированные риды по этому распределению.
- **Длина вставки / ампликона.** `merged` и `post_filter` — длины целых ампликонов без праймеров; они уже заложены в эталон (шаблон = ампликон). Для фрагментированной библиотеки длина вставки определяется отбором по длине (`SIZE_SELECTION_TARGET`, `SIZE_SELECTION_SD`), а не этими данными.
- **Перекрытие.** Показывает, сколько нуклеотидов R1 и R2 перекрывались в реальной паре; для PE150 на фрагментах 250 нт перекрытие ≈ 50 нт.
